# CNN Time Series Model Training on Google Colab

This notebook trains the **1D CNN** version of the multimodal deepfake model (`time_series_cnn_model.py`). It uses the same data (HDF5 embeddings) and training setup as the Transformer, but replaces the Transformer encoders with 1D convolutional encoders.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configure paths – point these to your files in Google Drive

HDF5_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/deepfake_embeddings_2.h5"
DRIVE_MODEL_PATH = "/content/drive/MyDrive/MIT/Lab/time_series_cnn_model.py"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/MIT/Lab"

print("="*60)
print("CONFIGURED PATHS (CNN model)")
print("="*60)
print(f"HDF5: {HDF5_FILE_PATH}")
print(f"CNN model: {DRIVE_MODEL_PATH}")
print(f"Checkpoints: {DRIVE_CHECKPOINT_DIR}")
print("="*60)

In [ ]:
# OPTION 3: Copy time_series_cnn_model.py from Google Drive
# The CNN module is self-contained (no time_series_model dependency).

import shutil, os

if os.path.exists(DRIVE_MODEL_PATH):
    os.makedirs("/content/models", exist_ok=True)
    shutil.copy(DRIVE_MODEL_PATH, "/content/models/time_series_cnn_model.py")
    print("✓ Copied time_series_cnn_model.py from Drive to /content/models/")
else:
    print(f"⚠️  Not found: {DRIVE_MODEL_PATH}")
    print("   Upload time_series_cnn_model.py to Drive and set DRIVE_MODEL_PATH.")

In [ ]:
# Install dependencies
!pip install h5py numpy scikit-learn tqdm matplotlib
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Enable GPU: Runtime > Change runtime type > GPU")

In [ ]:
# Patch paths in time_series_cnn_model.py and write modified script

import os, re

model_file = None
for p in ["/content/models/time_series_cnn_model.py", DRIVE_MODEL_PATH, "/content/itau-group4/time_series_cnn_model.py"]:
    if os.path.exists(p):
        model_file = p
        break

if model_file:
    with open(model_file, 'r') as f:
        content = f.read()
    content = content.replace("/Users/jerrysheng/Desktop/Lab/deepfake_embeddings_2.h5", HDF5_FILE_PATH)
    content = re.sub(r'save_dir\s*=\s*["\']\.\/checkpoints["\']', f'save_dir="{DRIVE_CHECKPOINT_DIR}"', content)
    out = "/content/models/time_series_cnn_model_modified.py"
    with open(out, 'w') as f:
        f.write(content)
    MODEL_FILE_TO_USE = out
    print(f"✓ Patched script: {out}")
else:
    MODEL_FILE_TO_USE = None
    print("⚠️  time_series_cnn_model.py not found")

In [ ]:
# Quick HDF5 check
import os
if os.path.exists(HDF5_FILE_PATH):
    print(f"✓ HDF5 found: {os.path.getsize(HDF5_FILE_PATH)/1e9:.2f} GB")
else:
    print(f"⚠️  HDF5 not found: {HDF5_FILE_PATH}")

In [ ]:
# Run CNN training in the background
# 💡 TIP: After starting training, you can now run cells 10-12 to monitor progress!

import subprocess, sys, os

# Global variable to store the training process
training_process = None

if MODEL_FILE_TO_USE and os.path.exists(MODEL_FILE_TO_USE):
    print("="*60)
    print("Starting CNN training in background...")
    print("="*60)
    print(f"Model: {MODEL_FILE_TO_USE}")
    print(f"HDF5: {HDF5_FILE_PATH}")
    print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")
    print("\n💡 Training is running in the background. You can now:")
    print("   • Run cell 9: Check if training process is running")
    print("   • Run cell 10: Check current progress (one-time snapshot)")
    print("   • Run cell 11: Real-time monitor (auto-refreshes every 30s)")
    print("   • Run cell 12: Quick status check (is training still running?)")
    print("="*60 + "\n")

    # Start training in background (non-blocking)
    training_process = subprocess.Popen(
        [sys.executable, MODEL_FILE_TO_USE],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    print(f"✅ Training started! Process ID: {training_process.pid}")
    print(f"💡 Run cell 9 to check if training is still running")
    print(f"💡 Run cell 10+ to monitor progress via training_history.json")
    print("="*60)
else:
    print("⚠️  Run the setup cells first (copy model, patch paths).")

## Training Progress Monitoring

**Run the cells below while training is in progress** (after starting cell 8) to monitor:
- Process status (is training still running?)
- Current epoch and latest metrics
- Training status (running/completed)
- Estimated time remaining

In [ ]:
# Check if training process is still running
import subprocess, os

print("="*60)
print("TRAINING PROCESS STATUS")
print("="*60)

# Check if training_process variable exists and is still running
try:
    if 'training_process' in globals() and training_process is not None:
        status = training_process.poll()
        if status is None:
            print(f"🟢 Training is RUNNING (PID: {training_process.pid})")
            print("💡 Process is active - training is in progress")
        else:
            print(f"🔴 Training process has FINISHED (exit code: {status})")
            if status == 0:
                print("✅ Training completed successfully!")
            else:
                print(f"⚠️  Training exited with error code {status}")
    else:
        print("⚠️  Training process not found in memory")
        print("💡 This is normal if:")
        print("   • You restarted the runtime")
        print("   • You haven't run cell 8 yet")
        print("   • Training already finished")
        print("\n💡 Use cell 11 (status check) to verify via training_history.json")
except NameError:
    print("⚠️  Training process variable not found")
    print("💡 Run cell 8 first to start training, or check cell 11 for file-based status")

print("\n" + "="*60)

In [ ]:
# Check current training progress (run this while training is active)
# This reads from training_history.json, so it works even if the process variable is lost
import json, os
from datetime import datetime

history_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'training_history.json')
checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_model.pt')

print("="*60)
print("TRAINING PROGRESS CHECK")
print("="*60)
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

if os.path.exists(history_file):
    try:
        with open(history_file) as f:
            history = json.load(f)
        epochs = history.get('epochs', [])
        if epochs:
            latest_epoch = epochs[-1]
            total_epochs = history.get('config', {}).get('num_epochs', len(epochs))
            print(f"📊 Current Epoch: {latest_epoch} / {total_epochs}")
            print(f"\nLatest Metrics (Epoch {latest_epoch}):")
            idx = len(epochs) - 1
            print(f"  Train - Loss: {history['train_loss'][idx]:.4f}, AUROC: {history['train_auroc'][idx]:.4f}, Acc: {history['train_accuracy'][idx]:.4f}")
            if history.get('val_loss') and len(history['val_loss']) > idx:
                print(f"  Val   - Loss: {history['val_loss'][idx]:.4f}, AUROC: {history['val_auroc'][idx]:.4f}, Acc: {history['val_accuracy'][idx]:.4f}")
            if history.get('best_epoch'):
                print(f"\n🏆 Best so far: Epoch {history['best_epoch']}, Val AUROC: {history['best_val_auroc']:.4f}")
            progress_pct = 100 * len(epochs) / total_epochs if total_epochs > 0 else 0
            print(f"\n📈 Progress: {len(epochs)}/{total_epochs} epochs ({progress_pct:.1f}%)")
        else:
            print("⚠️  History file exists but no epochs recorded yet")
    except Exception as e:
        print(f"⚠️  Error reading history: {e}")
else:
    print("⏳ Training not started yet or history file not created")
    print(f"   Expected at: {history_file}")

if os.path.exists(checkpoint_file):
    mtime = os.path.getmtime(checkpoint_file)
    age = (datetime.now().timestamp() - mtime) / 60
    print(f"\n✅ Checkpoint exists (last updated {age:.1f} min ago)")
else:
    print("\n⏳ No checkpoint yet (training may still be in progress)")

print("\n💡 Re-run this cell to refresh progress")
print("="*60)

In [ ]:
# Monitor training progress in real-time (auto-refresh every 30 seconds)
# Press STOP (interrupt kernel) to exit the monitoring loop

import json, os, time
from datetime import datetime
from IPython.display import clear_output

history_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'training_history.json')
checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_model.pt')
last_epoch_seen = -1
start_time = time.time()

print("="*60)
print("REAL-TIME TRAINING MONITOR")
print("="*60)
print("Press STOP (interrupt kernel) to exit monitoring\n")

try:
    while True:
        clear_output(wait=True)
        now = datetime.now().strftime('%H:%M:%S')
        elapsed = (time.time() - start_time) / 60

        if os.path.exists(history_file):
            try:
                with open(history_file) as f:
                    history = json.load(f)
                epochs = history.get('epochs', [])
                if epochs:
                    current_epoch = epochs[-1]
                    total_epochs = history.get('config', {}).get('num_epochs', len(epochs))
                    idx = len(epochs) - 1

                    print(f"[{now}] Elapsed: {elapsed:.1f} min | Epoch {current_epoch}/{total_epochs}")
                    print(f"Train: L={history['train_loss'][idx]:.4f} A={history['train_auroc'][idx]:.4f} Acc={history['train_accuracy'][idx]:.4f}")
                    if history.get('val_loss') and len(history['val_loss']) > idx:
                        print(f"Val:   L={history['val_loss'][idx]:.4f} A={history['val_auroc'][idx]:.4f} Acc={history['val_accuracy'][idx]:.4f}")
                    if history.get('best_epoch'):
                        print(f"Best: Epoch {history['best_epoch']}, Val AUROC {history['best_val_auroc']:.4f}")

                    if current_epoch > last_epoch_seen and last_epoch_seen >= 0:
                        epoch_time = elapsed / (current_epoch - last_epoch_seen) if current_epoch > last_epoch_seen else 0
                        remaining = (total_epochs - current_epoch) * epoch_time if epoch_time > 0 else 0
                        print(f"⏱️  Est. remaining: {remaining:.1f} min ({remaining/60:.1f} hours)")
                    last_epoch_seen = current_epoch

                    if current_epoch >= total_epochs:
                        print("\n✅ Training completed!")
                        break
                else:
                    print(f"[{now}] Waiting for first epoch...")
            except Exception as e:
                print(f"[{now}] Error: {e}")
        else:
            print(f"[{now}] Waiting for training to start...")

        time.sleep(30)  # Refresh every 30 seconds
except KeyboardInterrupt:
    print("\n\n⏹️  Monitoring stopped")

In [ ]:
# Quick status check: Is training still running?
# This method works even if you lost the process variable (e.g., after runtime restart)
import json, os, time
from datetime import datetime

history_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'training_history.json')
checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_model.pt')

print("="*60)
print("TRAINING STATUS CHECK (File-based)")
print("="*60)

if os.path.exists(history_file):
    mtime = os.path.getmtime(history_file)
    age_sec = time.time() - mtime
    age_min = age_sec / 60

    try:
        with open(history_file) as f:
            history = json.load(f)
        epochs = history.get('epochs', [])
        total_epochs = history.get('config', {}).get('num_epochs', 50)

        if epochs:
            current = epochs[-1]
            if current >= total_epochs:
                status = "✅ COMPLETED"
            elif age_min < 10:
                status = "🟢 RUNNING (likely)"
            else:
                status = f"⚠️  STALLED (no update in {age_min:.1f} min)"
            print(f"\nStatus: {status}")
            print(f"Current: Epoch {current}/{total_epochs}")
            print(f"History file last updated: {age_min:.1f} min ago")
            if current < total_epochs and age_min < 10:
                print("💡 Training appears active (recent file update)")
            elif current < total_epochs:
                print("⚠️  Training may have stopped (no recent update)")
                print("   • Check cell 9 for process status")
                print("   • Or check if Colab runtime disconnected")
        else:
            print("⏳ Training starting... (no epochs recorded yet)")
    except Exception as e:
        print(f"⚠️  Error reading history: {e}")
else:
    print("⏳ Training not started yet or history file not created")
    print(f"   Expected at: {history_file}")

print("="*60)

In [ ]:
# Locate and access your saved checkpoint (.pt file)
import os

print("="*60)
print("LOCATING YOUR CHECKPOINT FILE")
print("="*60)

checkpoint_dirs = [DRIVE_CHECKPOINT_DIR, "/content/models/checkpoints"]
found_checkpoints = False
checkpoint_files = []

for checkpoint_dir in checkpoint_dirs:
    if os.path.exists(checkpoint_dir):
        print(f"\n✓ Checkpoints directory found: {checkpoint_dir}")
        files = os.listdir(checkpoint_dir)
        if files:
            print(f"\n📁 Saved files ({len(files)}):")
            for f in files:
                filepath = os.path.join(checkpoint_dir, f)
                size = os.path.getsize(filepath) / 1e6
                print(f"  • {f} ({size:.2f} MB)" if f.endswith('.pt') else f"  • {f} ({size:.2f} KB)")
                if f.endswith('.pt'):
                    checkpoint_files.append((filepath, f))
            found_checkpoints = True
        else:
            print("  (no files found yet)")
        break

if found_checkpoints and checkpoint_files:
    print("\n" + "="*60)
    print("HOW TO ACCESS YOUR CHECKPOINT")
    print("="*60)
    drive_path = DRIVE_CHECKPOINT_DIR.replace("/content/drive", "")
    print(f"\n📍 Location: {drive_path}")
    print(f"💡 File persists in Drive after Colab session ends")
    for filepath, filename in checkpoint_files:
        if filename.endswith('.pt'):
            print(f"\n✅ Main checkpoint: {filename}\n   {filepath}")
else:
    print(f"\n⚠️  Checkpoints directory not found at {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# Download checkpoint to your local machine (optional)
from google.colab import files
import os

checkpoint_dir = DRIVE_CHECKPOINT_DIR
if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    if checkpoint_files:
        print(f"📥 Found {len(checkpoint_files)} checkpoint file(s):")
        for f in checkpoint_files:
            filepath = os.path.join(checkpoint_dir, f)
            print(f"  • {f} ({os.path.getsize(filepath)/1e6:.2f} MB)")
        print(f"\n💾 To download: uncomment and run the loop below (or use Drive directly)")
        # for f in checkpoint_files:
        #     files.download(os.path.join(checkpoint_dir, f))
    else:
        print("No .pt files found")
else:
    print(f"⚠️  Directory not found: {checkpoint_dir}")

## Notes

1. **GPU Runtime**: Select a GPU runtime (Runtime > Change runtime type > GPU)
2. **HDF5 File**: Upload `deepfake_embeddings_2.h5` to Drive and set `HDF5_FILE_PATH`
3. **Model File**: Upload `time_series_cnn_model.py` to Drive and set `DRIVE_MODEL_PATH`. Option 3 copies it. The CNN module is self-contained (no `time_series_model`).
4. **Checkpoints**: Saved to `DRIVE_CHECKPOINT_DIR` (`best_model.pt`, `training_history.json`). They persist after the Colab session.
5. **Training Time**: May take several hours depending on dataset size and epochs.

## Setup Checklist

- [ ] Mounted Google Drive
- [ ] Set `HDF5_FILE_PATH`, `DRIVE_MODEL_PATH`, `DRIVE_CHECKPOINT_DIR`
- [ ] Selected GPU runtime
- [ ] Run Option 3 to copy `time_series_cnn_model.py`

## Customization

Edit `CNNModelConfig` in `main()` of `time_series_cnn_model.py` (e.g. `num_conv_blocks`, `kernel_size`, `model_dim`).

## Testing

`python time_series_cnn_model.py test <hdf5_path> <checkpoint_path> [filter_dataset]`

In [ ]:
# Load and inspect the checkpoint (.pt file)
import torch
import os

print("="*60)
print("LOADING CHECKPOINT FILE")
print("="*60)

checkpoint_dir = DRIVE_CHECKPOINT_DIR
checkpoint_file = None
if os.path.exists(checkpoint_dir):
    pts = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    if pts:
        checkpoint_file = os.path.join(checkpoint_dir, pts[0])
        print(f"✓ Found: {pts[0]}")
    else:
        print("⚠️  No .pt files in checkpoint directory")
else:
    print(f"⚠️  Directory not found: {checkpoint_dir}")

if checkpoint_file and os.path.exists(checkpoint_file):
    print(f"\n📂 Loading: {checkpoint_file} ({os.path.getsize(checkpoint_file)/1e6:.2f} MB)")
    try:
        checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=False)
        print("\n" + "="*60 + "\nCHECKPOINT CONTENTS\n" + "="*60)
        print("\n📋 Keys:", list(checkpoint.keys()))
        if 'epoch' in checkpoint:
            print(f"📊 Epoch: {checkpoint['epoch']}")
        if 'val_auroc' in checkpoint:
            print(f"   Best Val AUROC: {checkpoint['val_auroc']:.4f}")
        if 'model_state_dict' in checkpoint:
            ms = checkpoint['model_state_dict']
            print(f"\n🧠 Model: {len(ms)} layers, {sum(p.numel() for p in ms.values()):,} params")
            for i, (n, p) in enumerate(list(ms.items())[:8]):
                print(f"   {n}: {tuple(p.shape)}")
            if len(ms) > 8:
                print(f"   ... and {len(ms)-8} more")
        if 'optimizer_state_dict' in checkpoint:
            o = checkpoint['optimizer_state_dict']
            if isinstance(o, dict) and 'param_groups' in o and o['param_groups']:
                print(f"\n⚙️  Optimizer lr: {o['param_groups'][0].get('lr', 'N/A')}")
        print("\n💡 Usage: model = AVTemporalModelCNN(config); model.load_state_dict(checkpoint['model_state_dict'])")
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n⚠️  Checkpoint not found. Run training first.")

In [ ]:
# Search for training logs / history files
import os, json, glob

print("="*60)
print("SEARCHING FOR TRAINING LOGS/HISTORY")
print("="*60)

locations = [DRIVE_CHECKPOINT_DIR, "/content/models", "/content"]
patterns = ["*.log", "*.txt", "*history*.json", "*training*.json", "*metrics*.json"]
found = []
for loc in locations:
    if os.path.exists(loc):
        for pat in patterns:
            found.extend(glob.glob(os.path.join(loc, "**", pat), recursive=True))

if found:
    print(f"\n✓ Found {len(found)} file(s):")
    for f in found:
        print(f"  • {f} ({os.path.getsize(f)/1024:.2f} KB)")
    for f in found:
        if f.endswith('.json'):
            try:
                with open(f) as fp:
                    d = json.load(fp)
                print(f"\n📄 {os.path.basename(f)}: keys={list(d.keys())[:12]}")
                if 'best_val_auroc' in d:
                    print(f"   best_val_auroc={d['best_val_auroc']:.4f}")
            except Exception as e:
                print(f"   Error: {e}")
else:
    print("\n⚠️  No log/history files found.")
    print("   time_series_cnn_model.py saves training_history.json to DRIVE_CHECKPOINT_DIR during training.")
print("\n" + "="*60)

In [ ]:
# Detailed checkpoint analysis
import torch, os

cf = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(cf):
    ck = torch.load(cf, map_location='cpu', weights_only=False)
    print("="*60 + "\nDETAILED CHECKPOINT ANALYSIS\n" + "="*60)
    for k, v in ck.items():
        if k == 'model_state_dict':
            print(f"  • {k}: dict, {len(v)} layers")
        elif k == 'optimizer_state_dict' and isinstance(v, dict) and 'state' in v:
            print(f"  • {k}: dict, {len(v['state'])} param states")
        elif k in ('epoch','val_auroc'):
            print(f"  • {k}: {v}")
    n = sum(p.numel() for p in ck['model_state_dict'].values())
    print(f"\n📊 Summary: epoch={ck.get('epoch','N/A')}, val_auroc={ck.get('val_auroc',0):.4f}, params={n:,}")
    print("\n💡 Per-epoch metrics are in training_history.json (saved by time_series_cnn_model).")
else:
    print("Checkpoint not found")

In [ ]:
# What's stored in the .pt checkpoint file
import torch, os

cf = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(cf):
    ck = torch.load(cf, map_location='cpu', weights_only=False)
    ms = ck['model_state_dict']
    total_opt_size = 0

    print("="*60 + "\nWHAT'S IN THE .PT FILE\n" + "="*60)
    print("\n📦 4 main components:\n")
    print("1️⃣  EPOCH – when best model was saved:", ck['epoch'])
    print("\n2️⃣  MODEL STATE DICT (trained weights)")
    print("   • Audio encoder (1D CNN), Video encoder (1D CNN), Fusion head (MLP)")
    print("   • Total params:", f"{sum(p.numel() for p in ms.values()):,}")
    ap = sum(p.numel() for n,p in ms.items() if 'audio_encoder' in n)
    vp = sum(p.numel() for n,p in ms.items() if 'video_encoder' in n)
    fp = sum(p.numel() for n,p in ms.items() if 'fusion_head' in n)
    tot = sum(p.numel() for p in ms.values())
    print("   • Audio CNN:", f"{ap:,} ({100*ap/tot:.1f}%)")
    print("   • Video CNN:", f"{vp:,} ({100*vp/tot:.1f}%)")
    print("   • Fusion:", f"{fp:,} ({100*fp/tot:.1f}%)")
    print("\n3️⃣  OPTIMIZER STATE – for resuming training")
    o = ck.get('optimizer_state_dict', {})
    if isinstance(o, dict) and 'state' in o:
        for ps in o['state'].values():
            for v in ps.values():
                if isinstance(v, torch.Tensor):
                    total_opt_size += v.numel() * 4
        print("   • Size ~{:.1f} MB".format(total_opt_size/1e6))
    print("\n4️⃣  VAL_AUROC – best validation AUROC:", f"{ck['val_auroc']:.4f}")

    print("\n" + "="*60 + "\n✅ Usage:")
    print("   model = AVTemporalModelCNN(config)")
    print("   model.load_state_dict(ck['model_state_dict'])")
    print("\n❌ Not stored: per-epoch loss/AUROC/accuracy (use training_history.json)")
else:
    print("Checkpoint not found")

In [ ]:
# Understanding your model's performance: AUROC
import torch, os

cf = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(cf):
    ck = torch.load(cf, map_location='cpu', weights_only=False)
    val_auroc = ck['val_auroc']
    nparams = sum(p.numel() for p in ck['model_state_dict'].values())

    print("="*60 + "\nUNDERSTANDING YOUR MODEL'S PERFORMANCE\n" + "="*60)
    print(f"\n📊 Validation AUROC: {val_auroc:.4f}")
    print("\n🎯 AUROC scale: 0.5=random, 0.7–0.8=acceptable, 0.8–0.9=good, 0.9–0.95=very good, 0.95–0.99=excellent")
    print(f"   {val_auroc:.4f} = {'🎉 Exceptional' if val_auroc>=0.99 else 'Excellent'}")
    print(f"\n📈 In practice: model ranks real vs fake correctly ~{val_auroc*100:.0f}% of the time")

    print("\n✅ Why it might be high:")
    print("   • 1D CNN + multimodal fusion fits sequential deepfake detection")
    print("   • Audio+video capture complementary signals")
    print("   • Model has ~{:,.0f} params; validation used for early stopping".format(nparams))
    print("   • AVDeepfake1M and ShareVeo3 are strong datasets")

    print("\n⚠️  Things to verify:")
    print("   • No data leakage (train/val split by video_id)")
    print("   • Test on a held-out set and other datasets")
    print("   • Check class balance and overfitting to val")

    print("\n✅ Next: test on held-out data, check precision/recall/F1, try other datasets.")
else:
    print("Checkpoint not found")

In [ ]:
# Load and visualize training history
import json, os
import matplotlib.pyplot as plt
import numpy as np

history_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'training_history.json')

if os.path.exists(history_file):
    with open(history_file) as f:
        history = json.load(f)
    epochs = history['epochs']
    print("="*60 + "\nLOADING TRAINING HISTORY\n" + "="*60)
    print(f"\n✓ Loaded: {history_file}")
    print(f"   Epochs: {len(epochs)}, Best: {history.get('best_epoch','N/A')}, Best Val AUROC: {history.get('best_val_auroc',0):.4f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('CNN Training History', fontsize=14, fontweight='bold')

    ax1 = axes[0,0]
    ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o', markersize=3)
    if history.get('val_loss'):
        ax1.plot(epochs, history['val_loss'], label='Val Loss', marker='s', markersize=3)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = axes[0,1]
    ax2.plot(epochs, history['train_auroc'], label='Train AUROC', marker='o', markersize=3)
    if history.get('val_auroc'):
        ax2.plot(epochs, history['val_auroc'], label='Val AUROC', marker='s', markersize=3)
        if history.get('best_epoch'):
            ax2.plot(history['best_epoch'], history['best_val_auroc'], 'r*', markersize=12, label=f"Best ({history['best_val_auroc']:.4f})")
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUROC'); ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1)

    ax3 = axes[1,0]
    ax3.plot(epochs, history['train_accuracy'], label='Train Acc', marker='o', markersize=3)
    if history.get('val_accuracy'):
        ax3.plot(epochs, history['val_accuracy'], label='Val Acc', marker='s', markersize=3)
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Accuracy'); ax3.legend(); ax3.grid(True, alpha=0.3); ax3.set_ylim(0, 1)

    ax4 = axes[1,1]
    ax4.axis('off')
    rows = [['Metric','Train','Val'], ['─'*12,'─'*10,'─'*10],
            ['Final Loss', f"{history['train_loss'][-1]:.4f}", f"{history['val_loss'][-1]:.4f}" if history.get('val_loss') else 'N/A'],
            ['Final AUROC', f"{history['train_auroc'][-1]:.4f}", f"{history['val_auroc'][-1]:.4f}" if history.get('val_auroc') else 'N/A'],
            ['Final Acc', f"{history['train_accuracy'][-1]:.4f}", f"{history['val_accuracy'][-1]:.4f}" if history.get('val_accuracy') else 'N/A']]
    if history.get('best_epoch'):
        rows += [['','',''], ['Best (epoch '+str(history['best_epoch'])+')','', f"{history['best_val_auroc']:.4f}"]]
    ax4.table(cellText=rows, cellLoc='left', loc='center', colWidths=[0.35,0.3,0.3])
    plt.tight_layout()
    plt.show()

    print("\n" + "="*60 + "\nPER-EPOCH (first 5 and last 3)\n" + "="*60)
    for i in [0,1,2,3,4] + ([len(epochs)-3, len(epochs)-2, len(epochs)-1] if len(epochs)>7 else []):
        if i >= len(epochs): continue
        tl=history['train_loss'][i]; ta=history['train_auroc'][i]; tac=history['train_accuracy'][i]
        vl=history['val_loss'][i] if history.get('val_loss') else 0; va=history['val_auroc'][i] if history.get('val_auroc') else 0; vac=history['val_accuracy'][i] if history.get('val_accuracy') else 0
        print(f"Epoch {epochs[i]:<4} Train L={tl:.4f} A={ta:.4f} Acc={tac:.4f}  Val L={vl:.4f} A={va:.4f} Acc={vac:.4f}")
else:
    print("⚠️  training_history.json not found at", history_file)
    print("   time_series_cnn_model.py saves it during training. Retrain if needed.")

## Quick Guide: training_history.json

`time_series_cnn_model.py` **already saves** `training_history.json` to `DRIVE_CHECKPOINT_DIR` each epoch during training. Run the **"Load and visualize training history"** cell above to plot loss, AUROC, and accuracy. No extra setup needed.

## Quick Access After Runtime Disconnect

If the runtime disconnected but training completed, run in order:

1. **Mount Drive** (cell 1)
2. **Configure paths** (cell 2)
3. **Locate checkpoint** (cell 9)
4. **Load and inspect checkpoint** (cell 12)
5. **Visualize training history** (cell 17) — if `training_history.json` exists

In [ ]:
# QUICK ACCESS: View results after runtime disconnect
# Run after re-mounting Drive and configuring paths

print("="*60)
print("QUICK ACCESS TO TRAINING RESULTS")
print("="*60)
print("\n📋 Run in order: 1) Mount Drive  2) Paths  3) Locate (cell 9)  4) Inspect (cell 12)  5) Visualize (cell 17)\n")
print("="*60)
print("QUICK CHECK")
print("="*60)

import os
cf = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")
hf = os.path.join(DRIVE_CHECKPOINT_DIR, "training_history.json")
if os.path.exists(cf):
    print(f"\n✅ Checkpoint: {cf} ({os.path.getsize(cf)/1e6:.2f} MB)")
else:
    print(f"\n⚠️  Checkpoint not found: {cf}")
if os.path.exists(hf):
    print(f"✅ History: {hf} ({os.path.getsize(hf)/1024:.2f} KB) → run cell 17 to plot")
else:
    print(f"⚠️  training_history.json not found (retrain if needed)")
print("\n" + "="*60)